NOTE - I tried to have ChatGPT be looser with categorizing elements into multiple attributes.

In [22]:
sgsg_df = pd.read_csv('transformed data/2-Uniqueness and Element avgs.csv')
sgsg_df

,O*NET-SOC Code,Job Title,AVG_TECH_SKILL_UNIQUENESS,AVG_TOOL_UNIQUENESS,AVG_TASK_UNIQUENESS,Mean T3 Uniqueness,Element Avg
0,11-1011.00,Chief Executives,1.285714,1.000,3.290,1.858571,2.442583
1,11-1011.03,Chief Sustainability Officers,1.000000,1.000,3.056,1.685333,2.210463
2,11-1021.00,General and Operations Managers,1.246575,1.278,2.765,1.763192,2.201566
3,11-2011.00,Advertising and Promotions Managers,1.397260,1.000,3.400,1.932420,2.054743
4,11-2021.00,Marketing Managers,1.150327,1.000,3.750,1.966776,2.122046
...,...,...,...,...,...,...,...
882,53-7071.00,Gas Compressor and Gas Pumping Station Operators,1.000000,1.647,2.692,1.779667,2.326854
883,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.000000,2.625,2.857,2.160667,2.255412
884,53-7073.00,Wellhead Pumpers,1.125000,2.768,2.714,2.202333,2.192687
885,53-7081.00,Refuse and Recyclable Material Collectors,1.400000,2.812,3.429,2.547000,1.863213


In [23]:
sgsg_df.nlargest(10, 'AVG_TASK_UNIQUENESS')

,O*NET-SOC Code,Job Title,AVG_TECH_SKILL_UNIQUENESS,AVG_TOOL_UNIQUENESS,AVG_TASK_UNIQUENESS,Mean T3 Uniqueness,Element Avg
87,13-2041.00,Credit Analysts,1.634146,1.000,4.182,2.272049,1.845770
618,45-4022.00,Logging Equipment Operators,1.375000,2.632,4.111,2.706000,2.179640
120,15-2021.00,Mathematicians,2.043011,1.143,4.000,2.395337,1.946903
634,47-2072.00,Pile Driver Operators,1.500000,2.400,4.000,2.633333,2.240702
408,29-1291.00,Acupuncturists,1.769231,3.162,3.833,2.921410,1.992862
838,53-1041.00,Aircraft Cargo Handling Supervisors,1.111111,2.625,3.833,2.523037,2.422716
409,29-1292.00,Dental Hygienists,1.210526,2.254,3.812,2.425509,2.124271
100,15-1221.00,Computer and Information Research Scientists,1.736196,2.394,3.800,2.643399,2.112062
440,31-1121.00,Home Health Aides,1.190476,1.469,3.800,2.153159,1.976936
661,47-4021.00,Elevator and Escalator Installers and Repairers,1.375000,1.300,3.800,2.158333,2.404207


# ChatGPT - Version 1 - No overlap - One cell

In [46]:
import pandas as pd

df_cognitive_complexity = pd.read_csv('elements sorted/gpt v1 no overlap/cognitive_complexity.csv')
df_creativity = pd.read_csv('elements sorted/gpt v1 no overlap/creativity.csv')
df_data_structure = pd.read_csv('elements sorted/gpt v1 no overlap/data_structure.csv')
df_decision_accountability = pd.read_csv('elements sorted/gpt v1 no overlap/decision_accountability.csv')
df_physical_requirements = pd.read_csv('elements sorted/gpt v1 no overlap/physical_requirements.csv')
df_routine_structure = pd.read_csv('elements sorted/gpt v1 no overlap/routine_structure.csv')
df_social_interactions = pd.read_csv('elements sorted/gpt v1 no overlap/social_interactions.csv')

dfs = [
    df_cognitive_complexity,
    df_creativity,
    df_data_structure,
    df_decision_accountability,
    df_physical_requirements,
    df_routine_structure,
    df_social_interactions
]



import pandas as pd
import numpy as np

ID_COLS = ["O*NET-SOC Code", "Title"]

# (optional but recommended) give your dfs names so the outputs are nicely labeled
df_names = [
    "cognitive_complexity",
    "creativity",
    "data_structure",
    "decision_accountability",
    "physical_requirements",
    "routine_structure",
    "social_interactions",
]

avg_dfs = {}   # name -> dataframe of averages
avg_list = []  # same outputs but as a list (if you prefer)

for name, df in zip(df_names, dfs):
    # safety: verify required cols exist
    missing = [c for c in ID_COLS if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")

    # choose columns to average: numeric columns excluding ID_COLS
    numeric_cols = df.drop(columns=ID_COLS, errors="ignore").select_dtypes(include="number").columns

    if len(numeric_cols) == 0:
        raise ValueError(f"{name} has no numeric columns to average (besides {ID_COLS}).")

    out = df[ID_COLS].copy()
    out[f"{name}_avg"] = df[numeric_cols].mean(axis=1, skipna=True)

    avg_dfs[name] = out
    avg_list.append(out)

for name, out in avg_dfs.items():
    globals()[f"df_{name}_avg"] = out

# Example: access one
# avg_dfs["creativity"].head()


avg_dfs = [
    df_cognitive_complexity_avg,
    df_creativity_avg,
    df_data_structure_avg,
    df_decision_accountability_avg,
    df_physical_requirements_avg,
    df_routine_structure_avg,
    df_social_interactions_avg
]


import pandas as pd
from functools import reduce

# Define the columns to merge on
merge_cols = ['O*NET-SOC Code', 'Title']

# Use reduce to merge all DataFrames sequentially
merged_df = reduce(lambda left, right: pd.merge(left, right, on=merge_cols, how='inner'), avg_dfs)

# Normalize for directionality -
# most of these have a negative impact on automation risk (meaning the score will be lower)
merged_df['physical_requirements_avg'] = 5 - merged_df['physical_requirements_avg']
merged_df['social_interactions_avg'] = 5 - merged_df['social_interactions_avg']
merged_df['cognitive_complexity_avg'] = 5 - merged_df['cognitive_complexity_avg']
merged_df['creativity_avg'] = 5 - merged_df['creativity_avg']
merged_df['decision_accountability_avg'] = 5 - merged_df['decision_accountability_avg']

# Apply weights according to my scale
merged_df['weighted_average_GPT_v1_non_overlap'] = (
    merged_df['cognitive_complexity_avg'] * 0.15 +
    merged_df['creativity_avg'] * 0.15 +
    merged_df['data_structure_avg'] * 0.10 +
    merged_df['decision_accountability_avg'] * 0.10 +
    merged_df['physical_requirements_avg'] * 0.15 +
    merged_df['routine_structure_avg'] * 0.20 +
    merged_df['social_interactions_avg'] * 0.15
)

# Rescale
from sklearn.preprocessing import MinMaxScaler
# Initialize the MinMaxScaler with a range of 1 to 5
scaler = MinMaxScaler(feature_range=(1, 5))
merged_df['GPT_V1_NoOverlap_Scaled'] = scaler.fit_transform(merged_df[['weighted_average_GPT_v1_non_overlap']])
# Average rescaled and non-rescaled
merged_df['GPT_V1_doubleweighed'] = (merged_df['weighted_average_GPT_v1_non_overlap'] + merged_df['GPT_V1_NoOverlap_Scaled']) / 2

# Rename df (saving this for last to maximize code reusability, not have to change dozens of names
gpt_v1_df = merged_df.copy()

gpt_v1_df.to_csv('transformed data-3-GPTv1 non overlap.csv', index=False)

gpt_v1_df

,O*NET-SOC Code,Title,cognitive_complexity_avg,creativity_avg,data_structure_avg,decision_accountability_avg,physical_requirements_avg,routine_structure_avg,social_interactions_avg,weighted_average_GPT_v1_non_overlap,GPT_V1_NoOverlap_Scaled,GPT_V1_doubleweighed
0,11-1011.00,Chief Executives,2.515545,2.543318,2.842733,0.805990,3.518718,2.732,1.524886,2.426642,1.672803,2.049723
1,11-1011.03,Chief Sustainability Officers,2.794998,2.502996,2.935123,1.719022,3.525297,2.008,1.897527,2.475137,1.889330,2.182234
2,11-1021.00,General and Operations Managers,2.879341,2.996607,2.639833,1.021243,3.413841,2.622,1.992559,2.582860,2.370301,2.476581
3,11-2011.00,Advertising and Promotions Managers,3.001268,2.702536,2.481961,1.745224,3.552578,2.522,2.162071,2.639887,2.624921,2.632404
4,11-2021.00,Marketing Managers,2.895341,2.738614,2.494049,1.708414,3.602884,2.568,1.948903,2.611707,2.499103,2.555405
...,...,...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.987875,3.379000,2.500994,0.918124,2.383196,3.274,2.689636,2.712668,2.949883,2.831275
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",3.000908,3.364275,2.218989,1.173527,2.483800,3.152,2.794480,2.716171,2.965524,2.840847
891,53-7073.00,Wellhead Pumpers,3.124872,3.504243,2.358553,1.327990,2.462484,2.826,2.760278,2.711636,2.945275,2.828455
892,53-7081.00,Refuse and Recyclable Material Collectors,3.478014,3.855050,1.284449,1.996345,2.607217,3.316,3.106563,2.948306,4.001987,3.475146


# Gemini - Version 1 - No overlap - One cell

In [49]:
import pandas as pd

df_cognitive_complexity = pd.read_csv('elements sorted/gemini v1 no overlap/cognitive_complexity.csv')
df_creativity = pd.read_csv('elements sorted/gemini v1 no overlap/creativity.csv')
df_data_structure = pd.read_csv('elements sorted/gemini v1 no overlap/data_structure.csv')
df_decision_accountability = pd.read_csv('elements sorted/gemini v1 no overlap/decision_accountability.csv')
df_physical_requirements = pd.read_csv('elements sorted/gemini v1 no overlap/physical_requirements.csv')
df_routine_structure = pd.read_csv('elements sorted/gemini v1 no overlap/routine_structure.csv')
df_social_interactions = pd.read_csv('elements sorted/gemini v1 no overlap/social_interactions.csv')

dfs = [
    df_cognitive_complexity,
    df_creativity,
    df_data_structure,
    df_decision_accountability,
    df_physical_requirements,
    df_routine_structure,
    df_social_interactions
]



import pandas as pd
import numpy as np

ID_COLS = ["O*NET-SOC Code", "Title"]

# (optional but recommended) give your dfs names so the outputs are nicely labeled
df_names = [
    "cognitive_complexity",
    "creativity",
    "data_structure",
    "decision_accountability",
    "physical_requirements",
    "routine_structure",
    "social_interactions",
]

avg_dfs = {}   # name -> dataframe of averages
avg_list = []  # same outputs but as a list (if you prefer)

for name, df in zip(df_names, dfs):
    # safety: verify required cols exist
    missing = [c for c in ID_COLS if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")

    # choose columns to average: numeric columns excluding ID_COLS
    numeric_cols = df.drop(columns=ID_COLS, errors="ignore").select_dtypes(include="number").columns

    if len(numeric_cols) == 0:
        raise ValueError(f"{name} has no numeric columns to average (besides {ID_COLS}).")

    out = df[ID_COLS].copy()
    out[f"{name}_avg"] = df[numeric_cols].mean(axis=1, skipna=True)

    avg_dfs[name] = out
    avg_list.append(out)

for name, out in avg_dfs.items():
    globals()[f"df_{name}_avg"] = out

# Example: access one
# avg_dfs["creativity"].head()


avg_dfs = [
    df_cognitive_complexity_avg,
    df_creativity_avg,
    df_data_structure_avg,
    df_decision_accountability_avg,
    df_physical_requirements_avg,
    df_routine_structure_avg,
    df_social_interactions_avg
]


import pandas as pd
from functools import reduce

# Define the columns to merge on
merge_cols = ['O*NET-SOC Code', 'Title']

# Use reduce to merge all DataFrames sequentially
merged_df = reduce(lambda left, right: pd.merge(left, right, on=merge_cols, how='inner'), avg_dfs)

# Normalize for directionality -
# most of these have a negative impact on automation risk (meaning the score will be lower)
merged_df['physical_requirements_avg'] = 5 - merged_df['physical_requirements_avg']
merged_df['social_interactions_avg'] = 5 - merged_df['social_interactions_avg']
merged_df['cognitive_complexity_avg'] = 5 - merged_df['cognitive_complexity_avg']
merged_df['creativity_avg'] = 5 - merged_df['creativity_avg']
merged_df['decision_accountability_avg'] = 5 - merged_df['decision_accountability_avg']

# Apply weights according to my scale
merged_df['weighted_average_Gemini_v1_non_overlap'] = (
    merged_df['cognitive_complexity_avg'] * 0.15 +
    merged_df['creativity_avg'] * 0.15 +
    merged_df['data_structure_avg'] * 0.10 +
    merged_df['decision_accountability_avg'] * 0.10 +
    merged_df['physical_requirements_avg'] * 0.15 +
    merged_df['routine_structure_avg'] * 0.20 +
    merged_df['social_interactions_avg'] * 0.15
)

# Rescale
from sklearn.preprocessing import MinMaxScaler
# Initialize the MinMaxScaler with a range of 1 to 5
scaler = MinMaxScaler(feature_range=(1, 5))
merged_df['Gemini_V1_NoOverlap_Scaled'] = scaler.fit_transform(merged_df[['weighted_average_Gemini_v1_non_overlap']])
# Average rescaled and non-rescaled
merged_df['Gemini_V1_doubleweighed'] = (merged_df['weighted_average_Gemini_v1_non_overlap'] + merged_df['Gemini_V1_NoOverlap_Scaled']) / 2


# Rename df (saving this for last to maximize code reusability, not have to change dozens of names
gemini_v1_df = merged_df.copy()

gemini_v1_df.to_csv('transformed data-3-Gemini v1 non overlap.csv', index=False)

gemini_v1_df

,O*NET-SOC Code,Title,cognitive_complexity_avg,creativity_avg,data_structure_avg,decision_accountability_avg,physical_requirements_avg,routine_structure_avg,social_interactions_avg,weighted_average_Gemini_v1_non_overlap,Gemini_V1_NoOverlap_Scaled,Gemini_V1_doubleweighed
0,11-1011.00,Chief Executives,2.467090,2.822797,2.817898,0.706326,3.583633,3.100000,1.562612,2.537842,1.818362,2.178102
1,11-1011.03,Chief Sustainability Officers,2.705213,2.781951,2.449030,1.460547,3.591119,2.465714,1.941748,2.537105,1.814934,2.176020
2,11-1021.00,General and Operations Managers,2.893803,3.205303,2.339782,1.144006,3.494753,2.960000,2.067406,2.689569,2.524144,2.606856
3,11-2011.00,Advertising and Promotions Managers,2.828308,2.938851,2.061210,1.727634,3.622828,2.892857,2.283343,2.708455,2.611999,2.660227
4,11-2021.00,Marketing Managers,2.785777,2.901171,2.181750,1.506147,3.655404,2.942857,2.082377,2.671071,2.438098,2.554584
...,...,...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,3.176847,3.474837,2.274999,1.373623,2.559462,3.378571,2.891265,2.855938,3.298042,3.076990
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",3.117762,3.367574,2.143631,1.504906,2.693883,3.197143,2.899213,2.816047,3.112481,2.964264
891,53-7073.00,Wellhead Pumpers,3.312860,3.558817,2.085364,1.650034,2.713714,3.010000,2.883189,2.845827,3.251007,3.048417
892,53-7081.00,Refuse and Recyclable Material Collectors,3.646789,3.931440,1.323999,2.389366,2.843741,3.170000,3.289857,3.062110,4.257088,3.659599


# Claude - Version 1 - No overlap - One cell

In [50]:
import pandas as pd

df_cognitive_complexity = pd.read_csv('elements sorted/claude v1 no overlap/cognitive_complexity.csv')
df_creativity = pd.read_csv('elements sorted/claude v1 no overlap/creativity.csv')
df_data_structure = pd.read_csv('elements sorted/claude v1 no overlap/data_structure.csv')
df_decision_accountability = pd.read_csv('elements sorted/claude v1 no overlap/decision_accountability.csv')
df_physical_requirements = pd.read_csv('elements sorted/claude v1 no overlap/physical_requirements.csv')
df_routine_structure = pd.read_csv('elements sorted/claude v1 no overlap/routine_structure.csv')
df_social_interactions = pd.read_csv('elements sorted/claude v1 no overlap/social_interactions.csv')

dfs = [
    df_cognitive_complexity,
    df_creativity,
    df_data_structure,
    df_decision_accountability,
    df_physical_requirements,
    df_routine_structure,
    df_social_interactions
]



import pandas as pd
import numpy as np

ID_COLS = ["O*NET-SOC Code", "Title"]

# (optional but recommended) give your dfs names so the outputs are nicely labeled
df_names = [
    "cognitive_complexity",
    "creativity",
    "data_structure",
    "decision_accountability",
    "physical_requirements",
    "routine_structure",
    "social_interactions",
]

avg_dfs = {}   # name -> dataframe of averages
avg_list = []  # same outputs but as a list (if you prefer)

for name, df in zip(df_names, dfs):
    # safety: verify required cols exist
    missing = [c for c in ID_COLS if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")

    # choose columns to average: numeric columns excluding ID_COLS
    numeric_cols = df.drop(columns=ID_COLS, errors="ignore").select_dtypes(include="number").columns

    if len(numeric_cols) == 0:
        raise ValueError(f"{name} has no numeric columns to average (besides {ID_COLS}).")

    out = df[ID_COLS].copy()
    out[f"{name}_avg"] = df[numeric_cols].mean(axis=1, skipna=True)

    avg_dfs[name] = out
    avg_list.append(out)

for name, out in avg_dfs.items():
    globals()[f"df_{name}_avg"] = out

# Example: access one
# avg_dfs["creativity"].head()


avg_dfs = [
    df_cognitive_complexity_avg,
    df_creativity_avg,
    df_data_structure_avg,
    df_decision_accountability_avg,
    df_physical_requirements_avg,
    df_routine_structure_avg,
    df_social_interactions_avg
]


import pandas as pd
from functools import reduce

# Define the columns to merge on
merge_cols = ['O*NET-SOC Code', 'Title']

# Use reduce to merge all DataFrames sequentially
merged_df = reduce(lambda left, right: pd.merge(left, right, on=merge_cols, how='inner'), avg_dfs)

# Normalize for directionality -
# most of these have a negative impact on automation risk (meaning the score will be lower)
merged_df['physical_requirements_avg'] = 5 - merged_df['physical_requirements_avg']
merged_df['social_interactions_avg'] = 5 - merged_df['social_interactions_avg']
merged_df['cognitive_complexity_avg'] = 5 - merged_df['cognitive_complexity_avg']
merged_df['creativity_avg'] = 5 - merged_df['creativity_avg']
merged_df['decision_accountability_avg'] = 5 - merged_df['decision_accountability_avg']

# Apply weights according to my scale
merged_df['weighted_average_Claude_v1_non_overlap'] = (
    merged_df['cognitive_complexity_avg'] * 0.15 +
    merged_df['creativity_avg'] * 0.15 +
    merged_df['data_structure_avg'] * 0.10 +
    merged_df['decision_accountability_avg'] * 0.10 +
    merged_df['physical_requirements_avg'] * 0.15 +
    merged_df['routine_structure_avg'] * 0.20 +
    merged_df['social_interactions_avg'] * 0.15
)

# Rescale
from sklearn.preprocessing import MinMaxScaler
# Initialize the MinMaxScaler with a range of 1 to 5
scaler = MinMaxScaler(feature_range=(1, 5))
merged_df['Claude_V1_NoOverlap_Scaled'] = scaler.fit_transform(merged_df[['weighted_average_Claude_v1_non_overlap']])
# Average rescaled and non-rescaled
merged_df['Claude_V1_doubleweighed'] = (merged_df['weighted_average_Claude_v1_non_overlap'] + merged_df['Claude_V1_NoOverlap_Scaled']) / 2


# Rename df (saving this for last to maximize code reusability, not have to change dozens of names
claude_v1_df = merged_df.copy()

claude_v1_df.to_csv('transformed data-3-Claude v1 non overlap.csv', index=False)

claude_v1_df

,O*NET-SOC Code,Title,cognitive_complexity_avg,creativity_avg,data_structure_avg,decision_accountability_avg,physical_requirements_avg,routine_structure_avg,social_interactions_avg,weighted_average_Claude_v1_non_overlap,Claude_V1_NoOverlap_Scaled,Claude_V1_doubleweighed
0,11-1011.00,Chief Executives,2.193079,2.937940,2.152163,0.845540,3.460524,2.179744,1.578832,2.261275,1.451239,1.856257
1,11-1011.03,Chief Sustainability Officers,2.469843,3.030226,1.914808,1.537000,3.440681,1.803207,1.896298,2.331380,1.885318,2.108349
2,11-1021.00,General and Operations Managers,2.710165,3.287321,1.887600,1.111377,3.311943,1.964175,2.050599,2.396737,2.290006,2.343371
3,11-2011.00,Advertising and Promotions Managers,2.705755,3.025112,1.749998,1.587171,3.509630,1.890987,2.280097,2.440004,2.557907,2.498955
4,11-2021.00,Marketing Managers,2.594651,3.045862,1.777900,1.415116,3.557630,1.883771,2.018331,2.378527,2.177249,2.277888
...,...,...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,3.077618,3.378395,2.094289,1.380851,2.199264,2.200389,2.813713,2.507940,2.978565,2.743253
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",3.111964,3.269145,2.040176,1.614734,2.379515,2.219165,2.833944,2.548509,3.229763,2.889136
891,53-7073.00,Wellhead Pumpers,3.246197,3.426505,1.929194,1.612879,2.310806,2.013830,2.861993,2.533799,3.138676,2.836237
892,53-7081.00,Refuse and Recyclable Material Collectors,3.623544,3.672771,1.503456,2.164631,2.493057,1.890201,3.261154,2.702428,4.182812,3.442620


# Put them all together

So it's an average, of averages, of averages, of averages.

In [81]:
#agg_df = claude_v1_df.copy()
from functools import reduce

# Merge dfs (add more if run additional models)
dfs_to_merge = [claude_v1_df, gemini_v1_df, gpt_v1_df]

# Merge the DataFrames on multiple columns using reduce()
agg_df = reduce(lambda left, right: pd.merge(left, right, on=['O*NET-SOC Code', 'Title'], how='inner'), dfs_to_merge)
agg_df = agg_df[['O*NET-SOC Code', 'Title', 'GPT_V1_doubleweighed', 'Claude_V1_doubleweighed', 'Gemini_V1_doubleweighed']]

# Select model score columns
selected_columns = agg_df[['GPT_V1_doubleweighed', 'Claude_V1_doubleweighed', 'Gemini_V1_doubleweighed']]

# Average all models
agg_df['Multi Model Consensus'] = selected_columns.mean(axis=1)

# Rank
agg_df['Consensus Rank'] = agg_df['Multi Model Consensus'].rank(ascending=False)

# Create categorial labels
# Define quantiles: 0-25% (low), 25-75% (mid), 75-100% (high)
# Note: qcut splits into 3 segments, so we need 4 points
quantiles = [0, 0.25, 0.75, 1.0]
labels = ['Low', 'Medium', 'High']
# Create the new categorical column
agg_df['Displacement Risk Category'] = pd.qcut(agg_df['Multi Model Consensus'], q=quantiles, labels=labels)

# Model disagreement
agg_df['Model Disagreement'] = selected_columns.max(axis=1) - selected_columns.min(axis=1)

agg_df

,O*NET-SOC Code,Title,GPT_V1_doubleweighed,Claude_V1_doubleweighed,Gemini_V1_doubleweighed,Multi Model Consensus,Consensus Rank,Displacement Risk Category,Model Disagreement
0,11-1011.00,Chief Executives,2.049723,1.856257,2.178102,2.028027,886.0,Low,0.321845
1,11-1011.03,Chief Sustainability Officers,2.182234,2.108349,2.176020,2.155534,870.0,Low,0.073885
2,11-1021.00,General and Operations Managers,2.476581,2.343371,2.606856,2.475603,705.0,Low,0.263485
3,11-2011.00,Advertising and Promotions Managers,2.632404,2.498955,2.660227,2.597195,616.0,Medium,0.161272
4,11-2021.00,Marketing Managers,2.555405,2.277888,2.554584,2.462626,716.0,Low,0.277518
...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.831275,2.743253,3.076990,2.883839,391.0,Medium,0.333737
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",2.840847,2.889136,2.964264,2.898083,377.0,Medium,0.123417
891,53-7073.00,Wellhead Pumpers,2.828455,2.836237,3.048417,2.904370,372.0,Medium,0.219962
892,53-7081.00,Refuse and Recyclable Material Collectors,3.475146,3.442620,3.659599,3.525789,54.0,High,0.216979


# What other models? Grok?

# Original working slow version

In [22]:
import pandas as pd

In [43]:
df_cognitive_complexity = pd.read_csv('elements sorted/gpt v1 no overlap/cognitive_complexity.csv')
df_creativity = pd.read_csv('elements sorted/gpt v1 no overlap/creativity.csv')
df_data_structure = pd.read_csv('elements sorted/gpt v1 no overlap/data_structure.csv')
df_decision_accountability = pd.read_csv('elements sorted/gpt v1 no overlap/decision_accountability.csv')
df_physical_requirements = pd.read_csv('elements sorted/gpt v1 no overlap/physical_requirements.csv')
df_routine_structure = pd.read_csv('elements sorted/gpt v1 no overlap/routine_structure.csv')
df_social_interactions = pd.read_csv('elements sorted/gpt v1 no overlap/social_interactions.csv')

In [44]:
dfs = [
    df_cognitive_complexity,
    df_creativity,
    df_data_structure,
    df_decision_accountability,
    df_physical_requirements,
    df_routine_structure,
    df_social_interactions
]

In [25]:
import pandas as pd
import numpy as np

ID_COLS = ["O*NET-SOC Code", "Title"]

# (optional but recommended) give your dfs names so the outputs are nicely labeled
df_names = [
    "cognitive_complexity",
    "creativity",
    "data_structure",
    "decision_accountability",
    "physical_requirements",
    "routine_structure",
    "social_interactions",
]

avg_dfs = {}   # name -> dataframe of averages
avg_list = []  # same outputs but as a list (if you prefer)

for name, df in zip(df_names, dfs):
    # safety: verify required cols exist
    missing = [c for c in ID_COLS if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing required columns: {missing}")

    # choose columns to average: numeric columns excluding ID_COLS
    numeric_cols = df.drop(columns=ID_COLS, errors="ignore").select_dtypes(include="number").columns

    if len(numeric_cols) == 0:
        raise ValueError(f"{name} has no numeric columns to average (besides {ID_COLS}).")

    out = df[ID_COLS].copy()
    out[f"{name}_avg"] = df[numeric_cols].mean(axis=1, skipna=True)

    avg_dfs[name] = out
    avg_list.append(out)

for name, out in avg_dfs.items():
    globals()[f"df_{name}_avg"] = out

# Example: access one
avg_dfs["creativity"].head()

,O*NET-SOC Code,Title,creativity_avg
0,11-1011.00,Chief Executives,2.456682
1,11-1011.03,Chief Sustainability Officers,2.497004
2,11-1021.00,General and Operations Managers,2.003393
3,11-2011.00,Advertising and Promotions Managers,2.297464
4,11-2021.00,Marketing Managers,2.261386


In [34]:
avg_dfs = [
    df_cognitive_complexity_avg,
    df_creativity_avg,
    df_data_structure_avg,
    df_decision_accountability_avg,
    df_physical_requirements_avg,
    df_routine_structure_avg,
    df_social_interactions_avg
]

In [35]:
import pandas as pd
from functools import reduce

# Define the columns to merge on
merge_cols = ['O*NET-SOC Code', 'Title']

# Use reduce to merge all DataFrames sequentially
merged_df = reduce(lambda left, right: pd.merge(left, right, on=merge_cols, how='inner'), avg_dfs)
merged_df

,O*NET-SOC Code,Title,cognitive_complexity_avg,creativity_avg,data_structure_avg,decision_accountability_avg,physical_requirements_avg,routine_structure_avg,social_interactions_avg
0,11-1011.00,Chief Executives,2.484455,2.456682,2.842733,4.194010,1.481282,2.732,3.475114
1,11-1011.03,Chief Sustainability Officers,2.205002,2.497004,2.935123,3.280978,1.474703,2.008,3.102473
2,11-1021.00,General and Operations Managers,2.120659,2.003393,2.639833,3.978757,1.586159,2.622,3.007441
3,11-2011.00,Advertising and Promotions Managers,1.998732,2.297464,2.481961,3.254776,1.447422,2.522,2.837929
4,11-2021.00,Marketing Managers,2.104659,2.261386,2.494049,3.291586,1.397116,2.568,3.051097
...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.012125,1.621000,2.500994,4.081876,2.616804,3.274,2.310364
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.999092,1.635725,2.218989,3.826473,2.516200,3.152,2.205520
891,53-7073.00,Wellhead Pumpers,1.875128,1.495757,2.358553,3.672010,2.537516,2.826,2.239722
892,53-7081.00,Refuse and Recyclable Material Collectors,1.521986,1.144950,1.284449,3.003655,2.392783,3.316,1.893437


Very dumb version for average. Not signed to account for differences in types.

In [39]:
merged_df['weighted_average_GPT_v1_non_overlap'] = (
    merged_df['cognitive_complexity_avg'] * 0.15 +
    merged_df['creativity_avg'] * 0.15 +
    merged_df['data_structure_avg'] * 0.10 +
    merged_df['decision_accountability_avg'] * 0.10 +
    merged_df['physical_requirements_avg'] * 0.15 +
    merged_df['routine_structure_avg'] * 0.20 +
    merged_df['social_interactions_avg'] * 0.15
)

merged_df

,O*NET-SOC Code,Title,cognitive_complexity_avg,creativity_avg,data_structure_avg,decision_accountability_avg,physical_requirements_avg,routine_structure_avg,social_interactions_avg,weighted_average_GPT_v1_non_overlap
0,11-1011.00,Chief Executives,2.484455,2.456682,2.842733,4.194010,1.481282,2.732,3.475114,2.734704
1,11-1011.03,Chief Sustainability Officers,2.205002,2.497004,2.935123,3.280978,1.474703,2.008,3.102473,2.415087
2,11-1021.00,General and Operations Managers,2.120659,2.003393,2.639833,3.978757,1.586159,2.622,3.007441,2.493907
3,11-2011.00,Advertising and Promotions Managers,1.998732,2.297464,2.481961,3.254776,1.447422,2.522,2.837929,2.365306
4,11-2021.00,Marketing Managers,2.104659,2.261386,2.494049,3.291586,1.397116,2.568,3.051097,2.414302
...,...,...,...,...,...,...,...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.012125,1.621000,2.500994,4.081876,2.616804,3.274,2.310364,2.597131
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.999092,1.635725,2.218989,3.826473,2.516200,3.152,2.205520,2.488427
891,53-7073.00,Wellhead Pumpers,1.875128,1.495757,2.358553,3.672010,2.537516,2.826,2.239722,2.390475
892,53-7081.00,Refuse and Recyclable Material Collectors,1.521986,1.144950,1.284449,3.003655,2.392783,3.316,1.893437,2.134984


In [ ]:
Routine - 20%

Cognitive complexity - 15%

Physical requirements - 15%

Social interactions - 15%

Creativity - 15%

Decision accountability 10%

Data structure 10%

In [28]:
df_cognitive_complexity_avg

,O*NET-SOC Code,Title,cognitive_complexity_avg
0,11-1011.00,Chief Executives,2.484455
1,11-1011.03,Chief Sustainability Officers,2.205002
2,11-1021.00,General and Operations Managers,2.120659
3,11-2011.00,Advertising and Promotions Managers,1.998732
4,11-2021.00,Marketing Managers,2.104659
...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2.012125
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.999092
891,53-7073.00,Wellhead Pumpers,1.875128
892,53-7081.00,Refuse and Recyclable Material Collectors,1.521986


In [30]:
df_creativity_avg

,O*NET-SOC Code,Title,creativity_avg
0,11-1011.00,Chief Executives,2.456682
1,11-1011.03,Chief Sustainability Officers,2.497004
2,11-1021.00,General and Operations Managers,2.003393
3,11-2011.00,Advertising and Promotions Managers,2.297464
4,11-2021.00,Marketing Managers,2.261386
...,...,...,...
889,53-7071.00,Gas Compressor and Gas Pumping Station Operators,1.621000
890,53-7072.00,"Pump Operators, Except Wellhead Pumpers",1.635725
891,53-7073.00,Wellhead Pumpers,1.495757
892,53-7081.00,Refuse and Recyclable Material Collectors,1.144950


# With overlaps
For this run, I permitted ChatGPT to assign an element into multiple categories, which will result in some being weighed twice. I decided against this initially but have come to consider that maybe it *should* matter more.